# Phase 1: Data Acquisition & Preprocessing (QC)
### Project: Pharmaco-OCD Clinical Decision Support System
**Student:** Final Year Undergraduate, Technology Management  
**Milestone 1:** Harmonised Dataset & Quality Control Report (End of Month 1)  

This notebook automates:
1. Downloading target transcriptomic cohorts from NCBI GEO (`GSE60190` OCD cohort & `GSE25974` SSRI response cohort).
2. Extracting the expression matrix and clinical metadata.
3. Inspecting distribution properties and applying $\log_2$ normalization.
4. Performing preliminary variance filtering and probe-to-gene mapping.

In [ ]:
# Install dependencies if running on Google Colab
# !pip install pandas numpy scipy statsmodels biopython GEOparse matplotlib seaborn

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to Python path
sys.path.append(os.path.abspath('../src'))
from data_loader import get_geo_series_url, download_file, parse_series_matrix
from preprocess import check_and_apply_log2, filter_low_variance_probes

DATA_RAW_DIR = '../data/raw'
DATA_PROCESSED_DIR = '../data/processed'
os.makedirs(DATA_RAW_DIR, exist_ok=True)
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
print("Environment initialized successfully.")

## 1. Download and Parse GSE60190 (OCD Cohort)

In [ ]:
gse_id = 'GSE60190'
url = get_geo_series_url(gse_id)
dest = os.path.join(DATA_RAW_DIR, f"{gse_id}_series_matrix.txt.gz")

download_file(url, dest)
df_expr_ocd, df_meta_ocd = parse_series_matrix(dest)

print(f"Dataset {gse_id} Loaded:")
print(f"• Probes: {df_expr_ocd.shape[0]}")
print(f"• Patient Samples: {df_expr_ocd.shape[1]}")
print("\nSample Metadata Preview:")
display(df_meta_ocd.head(3))

## 2. Quality Control & Log2 Normalization

In [ ]:
# Check distributions across first 5 samples
plt.figure(figsize=(10, 4))
sns.boxplot(data=df_expr_ocd.iloc[:, :5])
plt.title(f"{gse_id} Raw Expression Value Distribution (First 5 Samples)")
plt.ylabel("Expression Level")
plt.show()

# Apply log2 transformation if needed
df_expr_norm = check_and_apply_log2(df_expr_ocd)

plt.figure(figsize=(10, 4))
sns.boxplot(data=df_expr_norm.iloc[:, :5])
plt.title(f"{gse_id} Normalized Expression Distribution")
plt.ylabel("log2(Expression)")
plt.show()

## 3. Variance Filtering (Reducing Dimensionality)
Retain top 50% high-variance features to eliminate uninformative background noise.

In [ ]:
df_filtered = filter_low_variance_probes(df_expr_norm, top_percentile=0.5)
print(f"Dimensions after variance filtering: {df_filtered.shape}")